# JigNet jigsaw solver - demo
Solve one shuffled puzzle with the package code. Run from the repo root (`%cd` below) after training or downloading `jignet_weights.pth`.

In [ ]:
%cd ..
!pip install -q -r requirements.txt   # Colab only

In [ ]:
import random, cv2, torch, matplotlib.pyplot as plt
from data.tiling import make_puzzle, stitch
from models.checkpoint import load_jignet
from models.inference import predict_probs_matrix, prepare_tiles
from solver import build_pairwise_scores, solve_jigsaw_ilp, evaluate_board
from utils.constants import MEAN, STD

WEIGHTS = 'runs/jignet_exp2/last.pth'      # or jignet_weights.pth
IMAGE = 'dataset/jigsaw_dataset/images/test/img_1.png'
GRID = 4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_jignet(WEIGHTS, device)

In [ ]:
img = cv2.cvtColor(cv2.imread(IMAGE), cv2.COLOR_BGR2RGB)
tiles, gt = make_puzzle(img, GRID, 128, 'resize_tiles', shuffle=True, rng=random.Random(0))
probs = predict_probs_matrix(model, prepare_tiles(tiles, MEAN, STD, device))
res = solve_jigsaw_ilp(build_pairwise_scores(probs), GRID, time_limit=60)
exact, correct = evaluate_board(res.board, gt, GRID)
order = [res.board[r][c] for r in range(GRID) for c in range(GRID)]

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, im, t in zip(ax, [img, stitch(tiles, grid_size=GRID, gap=4), stitch(tiles, order, GRID)], ['original', 'shuffled', f'solved ({correct}/{GRID*GRID} tiles, exact={exact})']):
    a.imshow(im); a.set_title(t); a.axis('off')
plt.show()